# MediTask - Temporal Movement Intelligence (GRU / LSTM) - training on Google Colab

This notebook trains and compares the MediTask models on the harmonised rehabilitation dataset:

| Model | Description |
|---|---|
| A | Rule-based baseline (the app's thresholds) |
| A2 | DTW 1-nearest-neighbour on joint angles |
| B / C | LSTM / GRU on landmark coordinates only |
| **D** | **GRU hybrid: coordinates + biomechanical features (proposed)** |
| E | LSTM hybrid |

Outputs per repetition: exercise class, movement quality (correct / incorrect / incomplete) and per-frame phase.
Split: **participant-wise 70 / 10 / 20** (train / validation / test). The best model on *validation* quality macro-F1 is exported for the browser.

**Runtime → Change runtime type → GPU (T4)** before starting.

## 1. Get the project
Upload `MediTask.zip` to your Google Drive (e.g. `MyDrive/MediTask.zip`), or clone your GitHub repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/MediTask && unzip -q /content/drive/MyDrive/MediTask.zip -d /content/ && ls /content/MediTask
# or: !git clone https://github.com/<you>/MediTask.git /content/MediTask
%cd /content/MediTask/ml
!pip -q install -r requirements.txt

## 2. Datasets
Download each dataset from its official page **and accept its licence** (do not redistribute):

* KIMORE - https://vrai.dii.univpm.it/content/kimore-dataset
* UI-PRMD - https://webpages.uidaho.edu/ui-prmd/
* REHAB24-6 - see the dataset publication (Masaryk University); prepare `manifest.csv` as in `data/raw/rehab24_6/manifest.example.csv`
* MediTask custom recordings - JSON files exported from `collect.html`

Keep them in Drive and link them into `ml/data/raw/`. Before the first real run, **check every `VERIFY` note** in
`meditask_ml/adapters/public_datasets.py` and `config/label_taxonomy.json` against each dataset's README.

In [ ]:
DRIVE = '/content/drive/MyDrive/meditask_datasets'   # <- your folder
import os
for name in ['kimore', 'ui_prmd', 'rehab24_6', 'meditask_custom']:
    src, dst = f'{DRIVE}/{name}', f'data/raw/{name}'
    if os.path.exists(src):
        !rm -rf {dst} && ln -s {src} {dst}
        print('linked', name)
    else:
        print('missing', src)

## 3. Build the harmonised dataset and participant-wise splits

In [ ]:
!python -m meditask_ml.dataset --config config/datasets.json
import json; s = json.load(open('data/processed/dataset_summary.json'))
print(f"real independent sequences: {s['real_independent_sequences']} / target {s['target']}  (remaining {s['remaining_to_target']})")

> If you have no data yet, you can check the pipeline with **synthetic** data (results are meaningless):
> `!python -m meditask_ml.dataset --synthetic --out data/processed_synthetic` and pass `--data data/processed_synthetic` below.

## 4. Train and compare all models (≈ 10-30 min on a T4, depending on dataset size)

In [ ]:
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))
!python -m meditask_ml.run_experiments --epochs 100 --augment 2

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/results.md').read()))
import glob
for p in sorted(glob.glob('models/*/cm_quality.png')) + sorted(glob.glob('models/*/training_curves.png')):
    print(p); display(Image(p, width=520))

## 5. Single model / ablations (optional)

In [ ]:
# e.g. a smaller GRU for low-end phones, or more augmentation
!python -m meditask_ml.train --arch gru --features hybrid --units 64 --name gru_hybrid_64

## 6. Export for the web app and verify browser parity
`run_experiments` already exported the best model to `../models/web/`. Regenerate the parity fixtures so
`npm test` checks the JavaScript inference against Keras for *this* model.

In [ ]:
best = open('reports/selected_model.txt').read().strip(); print('selected:', best)
!python scripts/make_parity_fixtures.py --model models/{best}
!cd .. && zip -qr /content/drive/MyDrive/meditask_web_model.zip models/web ml/reports tests/parity/fixtures && echo saved to Drive

Copy `models/web/meditask_model.json` + `.bin` into your project, run `npm test`, and deploy (`firebase deploy`).
The patient app loads the model automatically and records per-repetition model outputs and latency in each session.

**Reporting:** quote test-set macro-F1 and per-class recall, the participant-wise split, the number of *independent* sequences
per source, and state that webcam measurements were not validated against clinical reference instruments.